In [1]:
!pip -q install pyspark datasets pyarrow pandas

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import pandas as pd
from datasets import load_dataset

In [4]:
# Change this to where you want everything saved
BASE_DIR = "/content/drive/MyDrive/7006SCN_temperature_project"
os.makedirs(BASE_DIR, exist_ok=True)

SAMPLE_PARQUET = os.path.join(BASE_DIR, "temperature_sample_100k.parquet")
METRICS_CSV    = os.path.join(BASE_DIR, "model_metrics.csv")
PRED_DIR       = os.path.join(BASE_DIR, "predictions")   # Spark will create a folder
MODEL_DIR      = os.path.join(BASE_DIR, "saved_models")  # Spark will create subfolders
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Saving outputs to:", BASE_DIR)

Saving outputs to: /content/drive/MyDrive/7006SCN_temperature_project


# 1) Download dataset + take 100,000 rows

In [5]:
ELEMENT = "TMAX"          # choose e.g. "TMAX", "TMIN", "TOBS", "PRCP"
N_SAMPLE = 100_000
SEED = 42

# Load HF dataset
ds = load_dataset("CatLikeIceCream/temperature", split="train")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


climate.csv:   0%|          | 0.00/2.73G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17097453 [00:00<?, ? examples/s]

In [6]:
# Filter to one element first (reduces data)
# Some datasets store ELEM as string; this filter is safe.
ds_elem = ds.filter(lambda x: x["ELEM"] == ELEMENT)

# Shuffle and take sample
ds_sample = ds_elem.shuffle(seed=SEED).select(range(N_SAMPLE))

# Save sample to Parquet on Google Drive (good for Spark)
ds_sample.to_parquet(SAMPLE_PARQUET)
print("Saved Parquet sample:", SAMPLE_PARQUET)

Filter:   0%|          | 0/17097453 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/100 [00:00<?, ?ba/s]

Saved Parquet sample: /content/drive/MyDrive/7006SCN_temperature_project/temperature_sample_100k.parquet


# 2) Start Spark (memory-safe config for Colab)

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("temperature-ml-colab")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.2


# 3) Read sample + build ML dataset

In [12]:
from functools import reduce

df = spark.read.parquet(SAMPLE_PARQUET)

# Show schema quickly
df.printSchema()

# ---- Feature/label design ----
# We’ll predict a mid-month day using earlier days:
# Features: DAY1..DAY14   Label: DAY15
# (You can change this if you want)
FEATURE_DAYS = [f"DAY{i}" for i in range(1, 15)]
LABEL_DAY = "DAY15"

# Keep only needed columns
# Some datasets include YEAR; if yours has it, we'll use it.
candidate_cols = ["MONTH", "ELEM"] + FEATURE_DAYS + [LABEL_DAY]
if "YEAR" in df.columns:
    candidate_cols.insert(0, "YEAR")

df = df.select([c for c in candidate_cols if c in df.columns])

# Clean / cast numeric columns
for c in FEATURE_DAYS + [LABEL_DAY, "MONTH"]:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

if "YEAR" in df.columns:
    df = df.withColumn("YEAR", F.col("YEAR").cast("double"))

# Filter: must have all features + label present
not_null_exprs = [F.col(c).isNotNull() for c in FEATURE_DAYS + [LABEL_DAY] if c in df.columns]
df = df.where(reduce(lambda a, b: a & b, not_null_exprs))

# Optional: remove obvious outliers if needed (comment out if you don't want)
# Example: temperature outliers (depends on units in dataset)
# df = df.where((F.col(LABEL_DAY) > -1000) & (F.col(LABEL_DAY) < 1000))

print("Rows after filtering:", df.count())

# Cache to avoid rereading Parquet repeatedly during training
df = df.persist()

root
 |-- ID: string (nullable = true)
 |-- YEAR: long (nullable = true)
 |-- MONTH: long (nullable = true)
 |-- ELEM: string (nullable = true)
 |-- DAY1: double (nullable = true)
 |-- DAY2: double (nullable = true)
 |-- DAY3: double (nullable = true)
 |-- DAY4: double (nullable = true)
 |-- DAY5: double (nullable = true)
 |-- DAY6: double (nullable = true)
 |-- DAY7: double (nullable = true)
 |-- DAY8: double (nullable = true)
 |-- DAY9: double (nullable = true)
 |-- DAY10: double (nullable = true)
 |-- DAY11: double (nullable = true)
 |-- DAY12: double (nullable = true)
 |-- DAY13: double (nullable = true)
 |-- DAY14: double (nullable = true)
 |-- DAY15: double (nullable = true)
 |-- DAY16: double (nullable = true)
 |-- DAY17: double (nullable = true)
 |-- DAY18: double (nullable = true)
 |-- DAY19: double (nullable = true)
 |-- DAY20: double (nullable = true)
 |-- DAY21: double (nullable = true)
 |-- DAY22: double (nullable = true)
 |-- DAY23: double (nullable = true)
 |-- DAY24: do

# 4) Build Spark ML pipeline components

In [10]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor
)

# Feature columns: include YEAR if present
feature_cols = FEATURE_DAYS.copy()
if "YEAR" in df.columns:
    feature_cols = ["YEAR"] + feature_cols
feature_cols = [c for c in feature_cols if c in df.columns]  # safety

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Train/test split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=SEED)
train_df = train_df.persist()
test_df = test_df.persist()

# Evaluators
evaluator_rmse = RegressionEvaluator(labelCol=LABEL_DAY, predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol=LABEL_DAY, predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol=LABEL_DAY, predictionCol="prediction", metricName="r2")


# 5) Train 4 models + evaluate + save results

In [13]:
models = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol=LABEL_DAY, maxIter=50, regParam=0.0),
    "DecisionTree":     DecisionTreeRegressor(featuresCol="features", labelCol=LABEL_DAY, maxDepth=10),
    "RandomForest":     RandomForestRegressor(featuresCol="features", labelCol=LABEL_DAY, numTrees=50, maxDepth=10, seed=SEED),
    "GBTRegressor":     GBTRegressor(featuresCol="features", labelCol=LABEL_DAY, maxIter=50, maxDepth=5, seed=SEED),
}

metrics_rows = []

for name, estimator in models.items():
    print(f"\n===== Training: {name} =====")
    pipeline = Pipeline(stages=[assembler, estimator])

    model = pipeline.fit(train_df)

    # Predict
    preds = model.transform(test_df).select(*(["MONTH"] + feature_cols + [LABEL_DAY, "prediction"]))

    rmse = evaluator_rmse.evaluate(preds)
    mae  = evaluator_mae.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)

    metrics_rows.append({
        "model": name,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "n_train": train_df.count(),
        "n_test": test_df.count(),
        "element": ELEMENT,
        "label": LABEL_DAY,
        "features": ",".join(feature_cols)
    })

    print(f"{name} -> RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

    # Save model
    out_model_path = os.path.join(MODEL_DIR, name)
    # Overwrite if exists
    try:
        import shutil
        if os.path.exists(out_model_path):
            shutil.rmtree(out_model_path)
    except Exception:
        pass
    model.write().overwrite().save(out_model_path)
    print("Saved model to:", out_model_path)

    # Save predictions (Parquet)
    out_pred_path = os.path.join(PRED_DIR, name)
    try:
        import shutil
        if os.path.exists(out_pred_path):
            shutil.rmtree(out_pred_path)
    except Exception:
        pass
    preds.write.mode("overwrite").parquet(out_pred_path)
    print("Saved predictions to:", out_pred_path)

# Save metrics to Drive (CSV)
metrics_df = pd.DataFrame(metrics_rows).sort_values("rmse")
metrics_df.to_csv(METRICS_CSV, index=False)
print("\nSaved metrics CSV:", METRICS_CSV)
display(metrics_df)


===== Training: LinearRegression =====
LinearRegression -> RMSE: 40.6375 | MAE: 29.9810 | R2: 0.8797
Saved model to: /content/drive/MyDrive/7006SCN_temperature_project/saved_models/LinearRegression
Saved predictions to: /content/drive/MyDrive/7006SCN_temperature_project/predictions/LinearRegression

===== Training: DecisionTree =====
DecisionTree -> RMSE: 42.7026 | MAE: 31.4183 | R2: 0.8671
Saved model to: /content/drive/MyDrive/7006SCN_temperature_project/saved_models/DecisionTree
Saved predictions to: /content/drive/MyDrive/7006SCN_temperature_project/predictions/DecisionTree

===== Training: RandomForest =====
RandomForest -> RMSE: 40.5732 | MAE: 30.2248 | R2: 0.8801
Saved model to: /content/drive/MyDrive/7006SCN_temperature_project/saved_models/RandomForest
Saved predictions to: /content/drive/MyDrive/7006SCN_temperature_project/predictions/RandomForest

===== Training: GBTRegressor =====
GBTRegressor -> RMSE: 40.9625 | MAE: 30.4556 | R2: 0.8777
Saved model to: /content/drive/MyDr

,model,rmse,mae,r2,n_train,n_test,element,label,features
2,RandomForest,40.573241,30.224828,0.880055,72519,18271,TMAX,DAY15,"YEAR,DAY1,DAY2,DAY3,DAY4,DAY5,DAY6,DAY7,DAY8,D..."
0,LinearRegression,40.637461,29.981025,0.879675,72519,18271,TMAX,DAY15,"YEAR,DAY1,DAY2,DAY3,DAY4,DAY5,DAY6,DAY7,DAY8,D..."
3,GBTRegressor,40.962464,30.455582,0.877743,72519,18271,TMAX,DAY15,"YEAR,DAY1,DAY2,DAY3,DAY4,DAY5,DAY6,DAY7,DAY8,D..."
1,DecisionTree,42.702594,31.418323,0.867135,72519,18271,TMAX,DAY15,"YEAR,DAY1,DAY2,DAY3,DAY4,DAY5,DAY6,DAY7,DAY8,D..."


# 6) Save the final filtered dataset

In [14]:
FILTERED_PARQUET_DIR = os.path.join(BASE_DIR, "filtered_training_data_parquet")
try:
    import shutil
    if os.path.exists(FILTERED_PARQUET_DIR):
        shutil.rmtree(FILTERED_PARQUET_DIR)
except Exception:
    pass

df.write.mode("overwrite").parquet(FILTERED_PARQUET_DIR)
print("Saved filtered ML-ready data to:", FILTERED_PARQUET_DIR)

Saved filtered ML-ready data to: /content/drive/MyDrive/7006SCN_temperature_project/filtered_training_data_parquet


# 7) Cleanup / unpersist

In [15]:
train_df.unpersist()
test_df.unpersist()
df.unpersist()

print("\nDONE")
print("Folder contains:")
print("- Sample Parquet:", SAMPLE_PARQUET)
print("- Filtered ML-ready Parquet folder:", FILTERED_PARQUET_DIR)
print("- Model metrics CSV:", METRICS_CSV)
print("- Saved models folder:", MODEL_DIR)
print("- Predictions folder:", PRED_DIR)


DONE
Folder contains:
- Sample Parquet: /content/drive/MyDrive/7006SCN_temperature_project/temperature_sample_100k.parquet
- Filtered ML-ready Parquet folder: /content/drive/MyDrive/7006SCN_temperature_project/filtered_training_data_parquet
- Model metrics CSV: /content/drive/MyDrive/7006SCN_temperature_project/model_metrics.csv
- Saved models folder: /content/drive/MyDrive/7006SCN_temperature_project/saved_models
- Predictions folder: /content/drive/MyDrive/7006SCN_temperature_project/predictions


In [16]:
SAMPLE_CSV = os.path.join(BASE_DIR, "temperature_sample_100k.csv")

# HuggingFace datasets supports direct CSV export
ds_sample.to_csv(SAMPLE_CSV, index=False)
print("Saved sample CSV:", SAMPLE_CSV)

Creating CSV from Arrow format:   0%|          | 0/100 [00:00<?, ?ba/s]

Saved sample CSV: /content/drive/MyDrive/7006SCN_temperature_project/temperature_sample_100k.csv


In [17]:
# -----------------------------------
# B) Save model performance (all algorithms) as CSV
# -----------------------------------
# (Your metrics_df is already created at the end of training.
#  This ensures it is saved as ONE CSV in Drive.)

METRICS_CSV = os.path.join(BASE_DIR, "model_metrics_all_algorithms.csv")
metrics_df.to_csv(METRICS_CSV, index=False)
print("Saved model performance CSV:", METRICS_CSV)


Saved model performance CSV: /content/drive/MyDrive/7006SCN_temperature_project/model_metrics_all_algorithms.csv
